In [ ]:
import torch
from torch import nn
import numpy as np
from string import Template
import math

import os 
import sys
sys.path.append(os.path.abspath("./"))
from gemm_layer_generator import generate_template_gemm
from codebooks_defs_generator import *



In [2]:
# My test settings (MVP)
N_LEARNERS = 1
CODEBOOK_SIZE = 8

M = 4
K = 4
N = 4
# M = number of rows of the input matrix X

# K = number of columns of the input matrix X
# = also the inner dot-product dimension

# N = number of rows of the weight matrix W
# = number of output features

TILE_L2_SIZE = 0
TILE_L1_SIZE = 0

SVE_LANES = 1

SAME_SEQ = True     # Specifies if the learners should have the same access sequence

In [3]:
gemm_0 = {
    "type": "gemm",
    "m": 4,
    "k": 4,
    "n": 4,
}

NN_structure = [gemm_0]

In [4]:
USE_BIAS = False

USE_F16 = False
USE_CODEBOOKS = True   # or False if you want NO_CODEBOOKS


# OUT_FOLDER = "./generated_headers/"
OUT_FOLDER = "./../gemm_definitions/"

generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES, USE_BIAS, USE_F16, SAME_SEQ, USE_CODEBOOKS) # Fix missing parameters in the function call


# Final torch network, one per learner
network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)] # Create a list of ModuleDicts, one for each learner, to hold the layers of each learner's network
# network = [
#     ModuleDict({}),
#     ModuleDict({}),
#     ModuleDict({})
# ]

for lay_cnt, layer in enumerate(NN_structure):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer["type"] == "gemm":
        m = layer["m"]
        k = layer["k"]
        n = layer["n"]

        in_shape = (m, k)
        out_shape = (m, n)

        gemm_weights, gemm_biases, input_matrix, golden_output = generate_template_gemm(
            SAME_SEQ,
            OUT_FOLDER + "gemm_data.h",
            lay_cnt,
            N_LEARNERS,
            CODEBOOK_SIZE,
            TILE_L1_SIZE,
            m,
            k,
            n,
            USE_F16,
            USE_CODEBOOKS,
            use_bias=USE_BIAS,
        )

        for learner in range(N_LEARNERS):
            linear = nn.Linear(k, n, bias=USE_BIAS)
            with torch.no_grad():
                linear.weight.copy_(torch.tensor(gemm_weights[learner], dtype=torch.float32))
                if USE_BIAS:
                    linear.bias.copy_(torch.tensor(gemm_biases[learner], dtype=torch.float32))
            network[learner][f"gemm_{lay_cnt}"] = linear

    else:
        print("ERROR!")
        exit(1)

    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()
    print("Input matrix:")
    print(input_matrix)
    # print("First input row:")
    # print(input_matrix[0])

[0] gemm
	 {'type': 'gemm', 'm': 4, 'k': 4, 'n': 4}
In shape: (4, 4)
Out shape: (4, 4)

Input matrix:
[[-0.25091976  0.90142864  0.4639879   0.19731697]
 [-0.6879627  -0.68801093 -0.88383275  0.7323523 ]
 [ 0.20223002  0.41614515 -0.958831    0.9398197 ]
 [ 0.6648853  -0.5753218  -0.6363501  -0.633191  ]]


In [5]:
input_tensor = torch.tensor(input_matrix, dtype=torch.float32)
print("input shape:", input_tensor.shape)  # should be [M, K]

for ens in range(N_LEARNERS):
    print(f"\n=============== LEARNER {ens} ===============\n")

    y = network[ens]["gemm_0"](input_tensor)     # only layer
    print("y shape:", y.shape)
    print("y:")
    print(y)

input shape: torch.Size([4, 4])

=============== LEARNER 0 ===============

y shape: torch.Size([4, 4])
y:
tensor([[-0.1608, -0.1235, -0.0610, -0.0628],
        [ 0.1663,  0.0885,  0.1475, -0.0210],
        [-0.1450, -0.0390,  0.0248, -0.1566],
        [ 0.1313,  0.1357, -0.0015,  0.0508]], grad_fn=<MmBackward0>)
